# 05 - Agregações Gold

## Objetivo

Este notebook constrói tabelas agregadas de consumo analítico a partir das dimensões e fatos já persistidos na camada Gold. Ele não altera a modelagem dimensional: apenas materializa visões recorrentes para vendas mensais, categorias, clientes e logística por UF.

Todas as saídas são gravadas em Delta com operação idempotente (`overwrite`) e reconciliadas contra as tabelas-fonte.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "datalake_mvp"
GOLD = "mvp_gold"

def gold_table(nome_tabela):
    return spark.table(f"{CATALOG}.{GOLD}.{nome_tabela}")

def salvar_gold(df, nome_tabela):
    destino = f"{CATALOG}.{GOLD}.{nome_tabela}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(destino)
    )
    print(f"✓ {destino} persistida.")

dim_customer = gold_table("dim_customer")
dim_product = gold_table("dim_product")
dim_date = gold_table("dim_date")
dim_order = gold_table("dim_order")
fact_order_items = gold_table("fact_order_items")
fact_payments = gold_table("fact_payments")

entradas = {
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_date": dim_date,
    "dim_order": dim_order,
    "fact_order_items": fact_order_items,
    "fact_payments": fact_payments
}

for nome, df in entradas.items():
    qtd = df.count()
    if qtd == 0:
        raise ValueError(f"Tabela de entrada {nome} está vazia.")
    print(f"{nome:<20} {qtd:>10,} registros")


## 1. Agregação mensal de vendas

Granularidade: uma linha por mês calendário com vendas registradas. Meses ausentes não são tratados como crescimento sequencial; o crescimento só é calculado quando existe o mês calendário imediatamente anterior.


In [ ]:
vendas_base = (
    fact_order_items.alias("f")
    .join(
        dim_date.select("date_key", "date").alias("d"),
        F.col("f.date_key") == F.col("d.date_key"),
        "inner"
    )
    .withColumn("mes_ref", F.trunc(F.col("d.date"), "month"))
    .withColumn("ano_mes", F.date_format(F.col("d.date"), "yyyy-MM"))
    .groupBy("mes_ref", "ano_mes")
    .agg(
        F.countDistinct("f.order_id").alias("pedidos"),
        F.count("*").alias("itens_vendidos"),
        F.round(F.sum("f.price"), 2).alias("valor_produtos"),
        F.round(F.sum("f.freight_value"), 2).alias("valor_frete"),
        F.round(F.sum("f.item_total_value"), 2).alias("valor_total")
    )
    .withColumn("ticket_medio_produtos", F.round(F.col("valor_produtos") / F.col("pedidos"), 2))
    .withColumn("itens_por_pedido", F.round(F.col("itens_vendidos") / F.col("pedidos"), 2))
)

mes_anterior = (
    vendas_base
    .select(
        F.add_months("mes_ref", 1).alias("mes_ref"),
        F.col("pedidos").alias("pedidos_mes_anterior"),
        F.col("ticket_medio_produtos").alias("ticket_mes_anterior"),
        F.col("valor_total").alias("valor_mes_anterior")
    )
)

agg_vendas_mensais = (
    vendas_base.alias("a")
    .join(mes_anterior.alias("b"), "mes_ref", "left")
    .withColumn(
        "crescimento_pedidos_pct",
        F.when(F.col("pedidos_mes_anterior") > 0,
               F.round((F.col("pedidos") - F.col("pedidos_mes_anterior")) / F.col("pedidos_mes_anterior") * 100, 2))
    )
    .withColumn(
        "crescimento_ticket_pct",
        F.when(F.col("ticket_mes_anterior") > 0,
               F.round((F.col("ticket_medio_produtos") - F.col("ticket_mes_anterior")) / F.col("ticket_mes_anterior") * 100, 2))
    )
    .withColumn(
        "crescimento_mensal_pct",
        F.when(F.col("valor_mes_anterior") > 0,
               F.round((F.col("valor_total") - F.col("valor_mes_anterior")) / F.col("valor_mes_anterior") * 100, 2))
    )
    .select(
        "mes_ref", "ano_mes", "pedidos", "itens_vendidos", "valor_produtos", "valor_frete",
        "valor_total", "ticket_medio_produtos", "itens_por_pedido",
        "crescimento_pedidos_pct", "crescimento_ticket_pct", "crescimento_mensal_pct"
    )
    .orderBy("mes_ref")
)

salvar_gold(agg_vendas_mensais, "agg_vendas_mensais")


In [ ]:
validacao_mensal = agg_vendas_mensais.agg(
    F.count("*").alias("meses"),
    F.countDistinct("mes_ref").alias("meses_distintos"),
    F.sum("itens_vendidos").alias("itens"),
    F.round(F.sum("valor_total"), 2).alias("receita")
).first()

fonte_mensal = fact_order_items.agg(
    F.count("*").alias("itens"),
    F.round(F.sum("item_total_value"), 2).alias("receita")
).first()

assert validacao_mensal["meses"] == validacao_mensal["meses_distintos"]
assert validacao_mensal["itens"] == fonte_mensal["itens"]
assert abs(float(validacao_mensal["receita"]) - float(fonte_mensal["receita"])) <= 0.01
print("✓ agg_vendas_mensais reconciliada com fact_order_items.")


## 2. Agregação por categoria

Granularidade: uma linha por categoria de produto. Categorias nulas são explicitamente agrupadas como `sem_categoria`.


In [ ]:
vendas_categoria = (
    fact_order_items.alias("f")
    .join(dim_product.select("product_id", "product_category_name").alias("p"), "product_id", "left")
    .withColumn("categoria", F.coalesce(F.col("p.product_category_name"), F.lit("sem_categoria")))
    .groupBy("categoria")
    .agg(
        F.countDistinct("order_id").alias("pedidos"),
        F.count("*").alias("itens_vendidos"),
        F.round(F.sum("price"), 2).alias("valor_produtos"),
        F.round(F.sum("freight_value"), 2).alias("valor_frete"),
        F.round(F.sum("item_total_value"), 2).alias("valor_total")
    )
    .withColumn("ticket_medio_produtos", F.round(F.col("valor_produtos") / F.col("pedidos"), 2))
    .withColumn("preco_medio_item", F.round(F.col("valor_produtos") / F.col("itens_vendidos"), 2))
    .withColumn("frete_medio", F.round(F.col("valor_frete") / F.col("itens_vendidos"), 2))
)

receita_total_categoria = vendas_categoria.agg(F.sum("valor_total").alias("v")).first()["v"]
window_categoria = Window.orderBy(F.desc("valor_total"), F.asc("categoria"))

agg_vendas_categoria = (
    vendas_categoria
    .withColumn("participacao_receita_pct", F.round(F.col("valor_total") / F.lit(receita_total_categoria) * 100, 2))
    .withColumn("ranking_receita", F.row_number().over(window_categoria))
    .orderBy("ranking_receita")
)

salvar_gold(agg_vendas_categoria, "agg_vendas_categoria")


In [ ]:
validacao_categoria = agg_vendas_categoria.agg(
    F.count("*").alias("categorias"),
    F.countDistinct("categoria").alias("categorias_distintas"),
    F.sum("itens_vendidos").alias("itens"),
    F.round(F.sum("valor_total"), 2).alias("receita")
).first()

fonte_categoria = fact_order_items.agg(
    F.count("*").alias("itens"),
    F.round(F.sum("item_total_value"), 2).alias("receita")
).first()

assert validacao_categoria["categorias"] == validacao_categoria["categorias_distintas"]
assert validacao_categoria["itens"] == fonte_categoria["itens"]
assert abs(float(validacao_categoria["receita"]) - float(fonte_categoria["receita"])) <= 0.01
print("✓ agg_vendas_categoria reconciliada com fact_order_items.")


## 3. Agregação por cliente único

Granularidade: uma linha por `customer_unique_id`. A recorrência é calculada pelo identificador estável do cliente, enquanto `customer_id` permanece como chave de relacionamento entre pedido e dimensão de clientes.


In [ ]:
base_clientes_pedidos = (
    dim_order.alias("o")
    .join(
        dim_customer.select("customer_id", "customer_unique_id").alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "inner"
    )
    .select("o.order_id", "o.order_purchase_timestamp", "c.customer_unique_id")
)

receita_por_pedido = (
    fact_order_items
    .groupBy("order_id")
    .agg(
        F.count("*").alias("quantidade_itens"),
        F.sum("price").alias("valor_produtos"),
        F.sum("freight_value").alias("valor_frete"),
        F.sum("item_total_value").alias("valor_pedido")
    )
)

base_cliente_financeira = (
    base_clientes_pedidos.alias("c")
    .join(receita_por_pedido.alias("r"), "order_id", "left")
)

agg_clientes = (
    base_cliente_financeira
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("quantidade_pedidos"),
        F.sum(F.coalesce(F.col("quantidade_itens"), F.lit(0))).alias("quantidade_itens"),
        F.sum("valor_produtos").alias("valor_produtos"),
        F.sum("valor_frete").alias("valor_frete"),
        F.sum("valor_pedido").alias("receita_total"),
        F.min("order_purchase_timestamp").alias("primeira_compra"),
        F.max("order_purchase_timestamp").alias("ultima_compra"),
        F.count("valor_pedido").alias("pedidos_com_valor")
    )
    .withColumn("ticket_medio", F.when(F.col("pedidos_com_valor") > 0, F.round(F.col("receita_total") / F.col("pedidos_com_valor"), 2)))
    .withColumn("flag_recorrente", F.when(F.col("quantidade_pedidos") > 1, 1).otherwise(0))
    .withColumn("dias_entre_primeira_ultima_compra", F.datediff(F.to_date("ultima_compra"), F.to_date("primeira_compra")))
)

salvar_gold(agg_clientes, "agg_clientes")


In [ ]:
validacao_clientes = agg_clientes.agg(
    F.count("*").alias("clientes"),
    F.countDistinct("customer_unique_id").alias("clientes_distintos"),
    F.sum("quantidade_pedidos").alias("pedidos"),
    F.sum("quantidade_itens").alias("itens"),
    F.round(F.sum("receita_total"), 2).alias("receita")
).first()

assert validacao_clientes["clientes"] == validacao_clientes["clientes_distintos"]
assert validacao_clientes["pedidos"] == dim_order.select("order_id").distinct().count()
assert validacao_clientes["itens"] == fact_order_items.count()
receita_fato = fact_order_items.agg(F.round(F.sum("item_total_value"),2).alias("v")).first()["v"]
assert abs(float(validacao_clientes["receita"]) - float(receita_fato)) <= 0.01
print("✓ agg_clientes reconciliada com dimensões e fato.")


## 4. Agregação logística por UF

Granularidade: uma linha por UF do cliente. As métricas de atraso são calculadas no nível de pedido, evitando multiplicação por itens.


In [ ]:
base_logistica_uf = (
    dim_order.alias("o")
    .join(dim_customer.select("customer_id", "customer_state").alias("c"), "customer_id", "left")
    .select(
        "o.order_id", F.col("c.customer_state").alias("uf"), "o.order_delivered_customer_date",
        "o.delivery_time_days", "o.delivery_delay_days", "o.flag_late_delivery"
    )
)

agg_logistica_uf = (
    base_logistica_uf
    .groupBy("uf")
    .agg(
        F.countDistinct("order_id").alias("pedidos"),
        F.sum(F.when(F.col("order_delivered_customer_date").isNotNull(), 1).otherwise(0)).alias("pedidos_entregues"),
        F.sum(F.when(F.col("flag_late_delivery") == 1, 1).otherwise(0)).alias("pedidos_atrasados"),
        F.round(F.avg("delivery_time_days"), 2).alias("tempo_medio_entrega_dias"),
        F.expr("percentile_approx(delivery_time_days, 0.5)").alias("mediana_entrega_dias"),
        F.expr("percentile_approx(delivery_time_days, 0.9)").alias("p90_entrega_dias"),
        F.round(F.avg(F.when(F.col("delivery_delay_days") > 0, F.col("delivery_delay_days"))), 2).alias("atraso_medio_quando_atrasado_dias")
    )
    .withColumn("taxa_atraso_pct", F.when(F.col("pedidos_entregues") > 0, F.round(F.col("pedidos_atrasados") / F.col("pedidos_entregues") * 100, 2)))
)

salvar_gold(agg_logistica_uf, "agg_logistica_uf")


In [ ]:
validacao_logistica = agg_logistica_uf.agg(
    F.count("*").alias("ufs"),
    F.countDistinct("uf").alias("ufs_distintas"),
    F.sum("pedidos").alias("pedidos"),
    F.sum("pedidos_entregues").alias("entregues"),
    F.sum("pedidos_atrasados").alias("atrasados")
).first()

fonte_logistica = dim_order.agg(
    F.count("*").alias("pedidos"),
    F.sum(F.when(F.col("order_delivered_customer_date").isNotNull(),1).otherwise(0)).alias("entregues"),
    F.sum(F.when(F.col("flag_late_delivery") == 1,1).otherwise(0)).alias("atrasados")
).first()

assert validacao_logistica["ufs"] == validacao_logistica["ufs_distintas"]
assert validacao_logistica["pedidos"] == fonte_logistica["pedidos"]
assert validacao_logistica["entregues"] == fonte_logistica["entregues"]
assert validacao_logistica["atrasados"] == fonte_logistica["atrasados"]
print("✓ agg_logistica_uf reconciliada com dim_order.")


## 5. Resumo das saídas

As quatro tabelas abaixo constituem a camada de consumo agregado do MVP e podem ser utilizadas por notebooks analíticos ou ferramentas de BI.


In [ ]:
for tabela in ["agg_vendas_mensais", "agg_vendas_categoria", "agg_clientes", "agg_logistica_uf"]:
    df = gold_table(tabela)
    print(f"{tabela:<25} {df.count():>10,} registros")

print("\n✓ Agregações Gold concluídas com sucesso.")
